In [1]:
import pandas as pd
import os
from nilearn import plotting, image, datasets
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import re
import ipywidgets as widgets
from IPython.display import display
from matplotlib.gridspec import GridSpec
from atlasreader import create_output
from matplotlib.colors import ListedColormap, BoundaryNorm
from nilearn.image import new_img_like
from scipy.ndimage import label


The Python package you are importing, AtlasReader, is licensed under the
BSD-3 license; however, the atlases it uses are separately licensed under more
restrictive frameworks.
By using AtlasReader, you agree to abide by the license terms of the
individual atlases. Information on these terms can be found online at:
https://github.com/miykael/atlasreader/tree/master/atlasreader/data



In [2]:
whichVersion = 'persentence' #persentence, 3sec, 6sec

virtueORvice = 'vices' # vices, virtues

#"21styear","tunnel","shapessocial","prettymouthaffair","prettymouthparanoia"

for whichStory in ["shapessocial","shapesphysical"]:

    print('run story: ',whichStory)
    
    for whichFoundation in ['1','2','3','4','5','6','7']:

        print('run foundation' ,whichFoundation)

        if whichFoundation == '1':
            FoundationName = 'deference'

        if whichFoundation == '2':
            FoundationName = 'fairness'

        if whichFoundation == '3':
            FoundationName = 'family'

        if whichFoundation == '4':
            FoundationName = 'group'

        if whichFoundation == '5':
            FoundationName = 'heroism'

        if whichFoundation == '6':
            FoundationName = 'property'

        if whichFoundation == '7':
            FoundationName = 'reciprocity'

        if whichVersion == "persentence":
            # output directory
            base_dir1 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/"
            )
        elif whichVersion == "3sec":
            # output directory
            base_dir1 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation_3sec/"
            )
        elif whichVersion == "6sec":
            # output directory
            base_dir1 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation_6sec/"
            )
            
        output_dir = Path(f"{base_dir1}{virtueORvice}/vsBaseline/{whichFoundation}/{whichStory}/")
    
        # make the output directory if it does not exist: 
        output_dir.mkdir(parents=True, exist_ok=True)
        
        # results directory
        
        if virtueORvice == 'virtues':
            virtueORviceDataFolder = '7_MAC'
        
        if virtueORvice == 'vices':
            virtueORviceDataFolder = '7_MAC_V'

        if whichVersion == "persentence":
            base_dir2 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_0/"
            )
        elif whichVersion == "3sec":
            base_dir2 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_3/"
            )
        elif whichVersion == "6sec":
            base_dir2 = os.path.expanduser(
                "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/processed_6/"
            )
            
        results_dir = Path(f"{base_dir2}{whichStory}/{virtueORviceDataFolder}/SecondLevel_contrast/")
        
        contrast_files = sorted(
            results_dir.glob(f"*foundation{whichFoundation}_VsBaseline_fdrcorrect_per_sentence_MAC*")
        )
        
        print(f"Found {len(contrast_files)} contrast maps.")
    
        for f in contrast_files:
            print(f)
            create_output(
                str(f),
                cluster_extent=2,
                direction="pos",
                outdir=str(output_dir)
            )
    
        stat_img = image.load_img(contrast_files[0])
        
        # Extract data and affine
        data = stat_img.get_fdata()
        affine = stat_img.affine
        
        # Label connected clusters
        labeled_array, num_features = label(data > 0)  # detects all nonzero clusters
        
        # Keep only clusters with ≥2 voxels
        filtered_data = np.zeros_like(data)
        for cluster_idx in range(1, num_features + 1):
            cluster_mask = labeled_array == cluster_idx
            if cluster_mask.sum() >= 2:
                filtered_data[cluster_mask] = data[cluster_mask]
        
        # Create filtered image
        filtered_img = image.new_img_like(stat_img, filtered_data)
        
        # Plot
        display = plotting.plot_glass_brain(
            filtered_img,
            colorbar=True,
            display_mode='lyrz',
            plot_abs=False,
            cmap='cold_hot',
            title='fdr corrected (clusters ≥ 2 voxels); ' + FoundationName
        )
        
        
        # Save as PNG
        display.savefig(str(output_dir)+'/glass_brain_clusters_min2vox.png', dpi=300)  
        
        display.close()

run story:  shapessocial
run foundation 1
Found 0 contrast maps.


IndexError: list index out of range